In [13]:
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import v2 as transforms
import librosa
from sklearn.metrics import roc_auc_score, mean_absolute_error
import glob
from tqdm import tqdm

In [2]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# class AudioDataset(torch.utils.data.Dataset):
#     def __init__(self, audio_dir, train):
#         self.audio_dir = audio_dir
#         file_list = os.listdir(audio_dir)

#         labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
#         self.labels = torch.tensor(labels, dtype=torch.int8).to(device)


#     def __len__(self):
#         return len(self.labels)

#     def __getitem__(self, idx):
#         return self.spect_dbs[idx], self.labels[idx]

In [5]:
def feature_extractor(file):
    def mfcc_extractor(file):
        data, sr = librosa.load(file)
        mfccs_features = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=128)
        mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)
        return mfccs_scaled_features

    def zero_extractor(file):
        data, sr = librosa.load(file)
        zeros = librosa.feature.zero_crossing_rate(data, frame_length=2048, hop_length=512, center=True)
        zeros_scaled_features = np.mean(zeros.T, axis=0)
        return zeros_scaled_features

    def rms_extractor(file):
        data, sr = librosa.load(file)
        rms = librosa.feature.rms(y=data)
        rms_scaled_features = np.mean(rms.T, axis=0)
        return rms_scaled_features

    def spectral_centroid_extractor(file):
        data, sr = librosa.load(file)
        sc = librosa.feature.spectral_centroid(y=data, sr=sr)
        sc_scaled_features = np.mean(sc.T, axis=0)
        return sc_scaled_features

    def spectral_bandwidth_extractor(file):
        data, sr = librosa.load(file)
        sb = librosa.feature.spectral_bandwidth(y=data, sr=sr)
        sb_scaled_features = np.mean(sb.T, axis=0)
        return sb_scaled_features

    def spectral_contrast_extractor(file):
        data, sr = librosa.load(file)
        sco = librosa.feature.spectral_contrast(y=data, sr=sr)
        sco_scaled_features = np.mean(sco.T, axis=0)
        return sco_scaled_features

    def polynomial_extractor(file):
        data, sr = librosa.load(file)
        poly = librosa.feature.poly_features(y=data, sr=sr, order=2)
        poly_scaled_features = np.mean(poly.T, axis=0)
        return poly_scaled_features
    mfcc_features = []
    zero_features = []
    rms_features = []
    sc_features = []
    sb_features = []
    sco_features = []
    poly_features = []
    labels = []
    for i in tqdm(file):
        label = 0 if i.split('/')[-1].startswith('n') else 1
        labels.append(label)

        mfcc = mfcc_extractor(i)
        mfcc_features.append(mfcc)

        zero = zero_extractor(i)
        zero_features.append(zero)

        rms = rms_extractor(i)
        rms_features.append(rms)

        spectral_centroid = spectral_centroid_extractor(i)
        sc_features.append(spectral_centroid)

        spectral_bandwidth = spectral_bandwidth_extractor(i)
        sb_features.append(spectral_bandwidth)

        spectral_contrast = spectral_contrast_extractor(i)
        sco_features.append(spectral_contrast)

        poly = polynomial_extractor(i)
        poly_features.append(poly)
    extracted_features_df = pd.DataFrame([mfcc_features, zero_features, rms_features, sc_features, sb_features, sco_features, poly_features, labels])
    extracted_features_df = extracted_features_df.T
    extracted_features_df.columns = ['mfcc', 'zero crossing rate', 'root mean square', 'spectral centroid', 'spectral bandwidth', 'spectral contrast', 'polynomial', 'labels']
    return extracted_features_df

In [6]:
datasets = {
    'train': glob.glob('archive/dev_data/dev_data/slider/train/*'),
    'test': glob.glob('archive/dev_data/dev_data/slider/test/*'),
    'additional_train': glob.glob('archive/eval_data/eval_data/slider/train/*'),
    'additional_test': glob.glob('archive/eval_data/eval_data/slider/test/*')
}

In [7]:
# df_train = feature_extractor(train_dataset)
# df_test = feature_extractor(test_dataset)

# labels_train = df_train.pop('labels')
# labels_test = df_test.pop('labels')

# df_train = df_train.applymap(lambda x: np.median(x))
# df_test = df_test.applymap(lambda x: np.median(x))

# for ds_name, globs in datasets.items():
#     df = feature_extractor(globs)
#     labels = df.pop('labels')
#     df = df.map(lambda x: np.median(x))
#     df['labels'] = labels
#     df.to_csv(f'new_dataset/{ds_name}_features.csv', index=False)

In [8]:
df_train = pd.read_csv('new_features/train_features.csv')
labels_train = df_train.pop('labels')

df_test = pd.read_csv('new_features/test_features.csv')
labels_test = df_test.pop('labels')

df_additional_train = pd.read_csv('new_features/additional_train_features.csv')
labels_additional_train = df_additional_train.pop('labels')

df_additional_test = pd.read_csv('new_features/additional_test_features.csv')
labels_additional_test = df_additional_test.pop('labels')

In [9]:
x = np.array(df_train)
x_t = np.array(df_test)

In [10]:
x.shape, x_t.shape

((2370, 7), (1101, 7))

In [11]:
from sklearn.preprocessing import StandardScaler, Normalizer
x = Normalizer().fit_transform(x)
x_t = Normalizer().fit_transform(x_t)
x = StandardScaler().fit_transform(x)
x_t = StandardScaler().fit_transform(x_t)

In [12]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(7, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, 16),
            nn.ELU(),
            nn.Linear(16, 8),
            nn.ELU(),
            nn.Linear(8, 4),
            nn.ELU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ELU(),
            nn.Linear(8, 16),
            nn.ELU(),
            nn.Linear(16, 32),
            nn.ELU(),
            nn.Linear(32, 64),
            nn.ELU(),
            nn.Linear(64, 7),
            nn.ELU()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
# Assume x and x_t are NumPy arrays with shape [num_samples, 7]
x_tensor = torch.tensor(x, dtype=torch.float32)
x_t_tensor = torch.tensor(x_t, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(x_tensor, x_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_t_tensor, x_t_tensor), batch_size=32, shuffle=False)

# Model, optimizer, loss
model = Autoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.05, patience=2)

In [17]:
num_epochs = 150

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch_x, _ in val_loader:
            output = model(batch_x)
            loss = criterion(output, batch_x)
            val_losses.append(loss.item())
            all_preds.append(output.numpy())
            all_targets.append(batch_x.numpy())

    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses)

    # Optional metrics
    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    mae = mean_absolute_error(targets, preds)
    try:
        msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
    except:
        msle = float("nan")

    print(f"Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - MAE: {mae:.4f} - MSLE: {msle:.4f}")

    # Adjust learning rate
    scheduler.step(avg_val_loss)

/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)


Epoch 1/150 - Train Loss: 0.1406 - Val Loss: 0.1884 - MAE: 0.2781 - MSLE: nan
Epoch 2/150 - Train Loss: 0.1312 - Val Loss: 0.1845 - MAE: 0.2772 - MSLE: nan
Epoch 3/150 - Train Loss: 0.1318 - Val Loss: 0.1826 - MAE: 0.2726 - MSLE: nan


/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)


Epoch 4/150 - Train Loss: 0.1280 - Val Loss: 0.1812 - MAE: 0.2689 - MSLE: nan
Epoch 5/150 - Train Loss: 0.1284 - Val Loss: 0.1801 - MAE: 0.2734 - MSLE: nan
Epoch 6/150 - Train Loss: 0.1261 - Val Loss: 0.1805 - MAE: 0.2692 - MSLE: nan


/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)


...

/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)


Epoch 144/150 - Train Loss: 0.1246 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan
Epoch 145/150 - Train Loss: 0.1221 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan
Epoch 146/150 - Train Loss: 0.1224 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan


/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)


Epoch 147/150 - Train Loss: 0.1221 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan
Epoch 148/150 - Train Loss: 0.1224 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan
Epoch 149/150 - Train Loss: 0.1230 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan
Epoch 150/150 - Train Loss: 0.1222 - Val Loss: 0.1775 - MAE: 0.2659 - MSLE: nan


/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: divide by zero encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
/tmp/ipykernel_2349/2141361438.py:34: RuntimeWarning: invalid value encountered in log1p
  msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)


In [18]:
def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).reshape(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors

In [20]:
errors = compute_reconstruction_errors(model, val_loader)
roc_auc_scores = roc_auc_score(labels_test, errors)
print(f"ROC AUC Score test: {roc_auc_scores}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)